In [ ]:
# Set Cover Notebook (alias-based, backend-agnostic test)

from optilab.aliases import Model, GRB, quicksum
import numpy as np

# -------------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------------
m = Model()
print("Backend:", m.backend_name)

# -------------------------------------------------------------------------------
# Problem data
# -------------------------------------------------------------------------------
rng = np.random.default_rng(42)

# Number of elements and sets
n_elements = rng.integers(20, 50)
n_sets = rng.integers(10, 25)

# Each set covers a random subset of elements
coverage = rng.integers(0, 2, size=(n_elements, n_sets))

# Ensure every element is covered by at least one set
for i in range(n_elements):
    if coverage[i].sum() == 0:
        coverage[i, rng.integers(0, n_sets)] = 1

# Cost of selecting each set
cost = rng.integers(1, 20, size=n_sets)

print("n elements:", n_elements)
print("n sets:", n_sets)
print("cost per set:", cost)

# -------------------------------------------------------------------------------
# Decision variables
# -------------------------------------------------------------------------------
x = [m.add_binary_var(name=f"x_{j}") for j in range(n_sets)]

# -------------------------------------------------------------------------------
# Constraint
# -------------------------------------------------------------------------------

# Every element must be covered by at least one selected set
for i in range(n_elements):
    m.add_constraint(
        quicksum(coverage[i][j] * x[j] for j in range(n_sets)) >= 1
    )

# -------------------------------------------------------------------------------
# Objective
# -------------------------------------------------------------------------------
m.set_objective(
    quicksum(cost[j] * x[j] for j in range(n_sets)),
    GRB.MINIMIZE
)

# -------------------------------------------------------------------------------
# Solve
# -------------------------------------------------------------------------------
m.solve()

# -------------------------------------------------------------------------------
# Extract solution
# -------------------------------------------------------------------------------
solution = np.array([v.x for v in x])

print(f"Solution:     {solution}")
print("Total cost:", float(np.sum(cost * solution)))

# -------------------------------------------------------------------------------
# Coverage check
# -------------------------------------------------------------------------------
covered = np.zeros(n_elements)
for i in range(n_elements):
    covered[i] = np.sum(coverage[i] * solution)

print("All elements covered:", np.all(covered >= 1))